# Anomaly Clustering

Cluster anomalies detected by Transformer+OC-SVM, PNN (spoofing gain), and PRAE (RFDR).

In [1]:
import os
import sys
import glob
import json
import logging
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, os.path.abspath(".."))

from detection.data.loaders import create_sequences, load_processed
from detection.data.preprocessing import get_time_frac, assign_period
from detection.models import hybrid, pnn as pnn_module, prae as prae_module
from detection.models.transformer import BottleneckTransformer
from detection.spoofing.gain import compute_spoofing_gains_batch
from detection.thresholds.rfdr import RollingFalseDiscoveryRate
from detection.trainers.factory import load_model

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
TRAIN_YEARS = [2015, 2017]
# Subsample mode: set to None to use full data
MAX_ROWS = None  # rows per file (must be > SEQ_LENGTH)

DATA_DIR = os.path.join("..", "data", "processed", "TOTF.PA-book")
MODEL_TYPES = ["transformer_ocsvm", "pnn", "prae"]

FILES_2010 = sorted(glob.glob(os.path.join(DATA_DIR, "2010-*.parquet")))
FILES_2015 = sorted(glob.glob(os.path.join(DATA_DIR, "2015-*.parquet")))
FILES_2017 = sorted(glob.glob(os.path.join(DATA_DIR, "2017-*.parquet")))
TEST_FILES = [FILES_2010[0], FILES_2010[1], FILES_2010[2], FILES_2010[-3], FILES_2010[-2], FILES_2010[-1],
                FILES_2015[-3], FILES_2015[-2], FILES_2015[-1], 
                FILES_2017[-6], FILES_2017[-5], FILES_2017[-4], FILES_2017[-3], FILES_2017[-2], FILES_2017[-1]]

SEQ_LENGTH = 25
BATCH_SIZE = 64

LOB_COLUMNS = [
    f"{side}-{typ}-{lvl}"
    for lvl in range(1, 11)
    for side, typ in [
        ("bid", "price"),
        ("bid", "volume"),
        ("ask", "price"),
        ("ask", "volume"),
    ]
]

SPOOF_Q = 4500
SPOOF_q = 100
SPOOF_DELTA_A = 0.0
SPOOF_DELTA_B = 0.01
SPOOF_FEES = {"maker": 0.0, "taker": 0.05}

RFDR_WINDOW = 500
RFDR_ALPHA = 0.05

PERIODS = {
    "1st_hour": (9.0, 10.0),
    "rest_of_morning": (10.0, 12.0),
    "afternoon": (12.0, 15.5),
    "american_open": (15.5, 17.5),
}

HDBSCAN_MIN_CLUSTER_FRAC = 20  # min_cluster_size = max(5, n_anom // this)
HDBSCAN_MAX_CLUSTER_SIZE = 500  # hard cap to avoid OOM on large anomaly sets
HDBSCAN_MIN_SAMPLES = 15       # explicit min_samples, kept small to avoid OOM
RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# Per-file cache helpers
# ---------------------------------------------------------------------------
def _cache_dir(year):
    return os.path.join("..", "results", str(year), "test_output", "cache")

def _cache_path(year, basename):
    return os.path.join(_cache_dir(year), f"{basename}.npz")

def _is_file_cached(year, basename):
    return os.path.exists(_cache_path(year, basename)) and MAX_ROWS is None

def _migrate_monolithic_cache(year):
    """Extract per-file NPZ caches from the old monolithic format (if present)."""
    output_dir = os.path.join("..", "results", str(year), "test_output")
    meta_path = os.path.join(output_dir, "test_meta.json")
    if not os.path.exists(meta_path):
        return 0
    with open(meta_path) as f:
        meta = json.load(f)
    day_names = meta.get("day_names", [])
    boundaries = meta.get("day_boundaries", [])
    if len(boundaries) < len(day_names) + 1:
        return 0
    cache_d = _cache_dir(year)
    if all(os.path.exists(os.path.join(cache_d, f"{dn}.npz")) for dn in day_names):
        return 0  # already migrated
    try:
        full_scores = {mt: np.load(os.path.join(output_dir, f"{mt}_scores.npy")) for mt in MODEL_TYPES}
        full_preds = {mt: np.load(os.path.join(output_dir, f"{mt}_preds.npy")) for mt in MODEL_TYPES}
    except FileNotFoundError:
        return 0
    os.makedirs(cache_d, exist_ok=True)
    migrated = 0
    for i, dn in enumerate(day_names):
        npz_path = os.path.join(cache_d, f"{dn}.npz")
        if os.path.exists(npz_path):
            continue
        lo, hi = boundaries[i], boundaries[i + 1]
        data = {}
        for mt in MODEL_TYPES:
            data[f"{mt}_scores"] = full_scores[mt][lo:hi]
            data[f"{mt}_preds"] = full_preds[mt][lo:hi]
        np.savez(npz_path, **data)
        migrated += 1
    return migrated

# Run migration and report cache status
for _y in TRAIN_YEARS:
    n_migrated = _migrate_monolithic_cache(_y)
    if n_migrated > 0:
        logger.info(f"Migrated {n_migrated} files from monolithic cache for year {_y}.")

print("Cache status per year:")
for _y in TRAIN_YEARS:
    _bns = [os.path.basename(f) for f in TEST_FILES]
    _cached = [bn for bn in _bns if _is_file_cached(_y, bn)]
    _missing = [bn for bn in _bns if not _is_file_cached(_y, bn)]
    print(f"  {_y}: {len(_cached)}/{len(_bns)} cached, {len(_missing)} to compute")
    for m in _missing[:5]:
        print(f"       - {m}")
    if len(_missing) > 5:
        print(f"       ... and {len(_missing) - 5} more")

Cache status per year:
  2015: 0/15 cached, 15 to compute
       - 2010-01-01-TOTF.PA-book.parquet
       - 2010-01-04-TOTF.PA-book.parquet
       - 2010-01-05-TOTF.PA-book.parquet
       - 2010-12-29-TOTF.PA-book.parquet
       - 2010-12-30-TOTF.PA-book.parquet
       ... and 10 more
  2017: 0/15 cached, 15 to compute
       - 2010-01-01-TOTF.PA-book.parquet
       - 2010-01-04-TOTF.PA-book.parquet
       - 2010-01-05-TOTF.PA-book.parquet
       - 2010-12-29-TOTF.PA-book.parquet
       - 2010-12-30-TOTF.PA-book.parquet
       ... and 10 more


In [3]:
feature_names_by_year = {}
loaded_models_by_year = {}
loaded_scalers_by_year = {}

for year in TRAIN_YEARS:
    results_dir = os.path.join("..", "results", str(year))

    # Load feature names
    feat_map = {}
    for mt in MODEL_TYPES:
        feat_path = os.path.join(results_dir, f"{mt}_features.txt")
        if os.path.exists(feat_path):
            with open(feat_path) as fh:
                feat_map[mt] = [ln.strip() for ln in fh if ln.strip()]
        else:
            _, tmp = load_processed(TEST_FILES[0], "xltime", LOB_COLUMNS)
            feat_map[mt] = tmp.columns.tolist()
    feature_names_by_year[year] = feat_map

    # Only load models if some files need computation
    _bns = [os.path.basename(f) for f in TEST_FILES]
    needs_compute = any(not _is_file_cached(year, bn) for bn in _bns)

    if needs_compute:
        models = {}
        scalers = {}
        for model_type in MODEL_TYPES:
            feat_names = feat_map[model_type]
            weights_path = os.path.join(results_dir, f"{model_type}_weights.pth")
            model, ocsvm = load_model(model_type, len(feat_names), weights_path, DEVICE, SEQ_LENGTH)
            models[model_type] = (model, ocsvm)

            scaler_path = os.path.join(results_dir, f"{model_type}_scaler.pkl")
            scalers[model_type] = (
                joblib.load(scaler_path) if os.path.exists(scaler_path) else MinMaxScaler()
            )
        loaded_models_by_year[year] = models
        loaded_scalers_by_year[year] = scalers
        print(f"Year {year}: models & scalers loaded (computation needed).")
    else:
        print(f"Year {year}: all files cached, skipping model loading.")

Year 2015: models & scalers loaded (computation needed).
Year 2017: models & scalers loaded (computation needed).


In [ ]:
def _load_and_slice(filepath, time_col, lob_cols, max_rows=None):
    df, feats = load_processed(filepath, time_col, lob_cols)
    if max_rows is not None:
        df = df.iloc[:max_rows].reset_index(drop=True)
        feats = feats.iloc[:max_rows].reset_index(drop=True)
    return df, feats


def _compute_file_scores(year, features_day, spread_raw):
    """Compute scores and preds for one file using one year's models."""
    feat_map = feature_names_by_year[year]
    models = loaded_models_by_year[year]
    scalers = loaded_scalers_by_year[year]
    results = {}

    for model_type in MODEL_TYPES:
        feat_names = feat_map[model_type]
        scaler = scalers[model_type]
        model, ocsvm = models[model_type]

        feat_df = features_day.copy()
        for col in feat_names:
            if col not in feat_df.columns:
                feat_df[col] = 0.0
        feat_df = feat_df[feat_names]

        scaled = scaler.transform(feat_df.values.astype(np.float32)).astype(np.float32)
        sequences = create_sequences(scaled, SEQ_LENGTH)

        if model_type == "transformer_ocsvm":
            x_tensor = torch.tensor(sequences, dtype=torch.float32)
            loader = DataLoader(
                TensorDataset(x_tensor, x_tensor), batch_size=BATCH_SIZE, shuffle=False
            )
            if ocsvm is not None:
                detector = hybrid.TransformerOCSVM.__new__(hybrid.TransformerOCSVM)
                detector.transformer = model
                detector.ocsvm = ocsvm
                scores = detector.predict(loader)
            else:
                scores_list = []
                with torch.no_grad():
                    for batch in loader:
                        x = batch[0].to(DEVICE)
                        rec = model(x)
                        scores_list.append(
                            torch.mean((x - rec) ** 2, dim=(1, 2)).cpu().numpy()
                        )
                scores = np.concatenate(scores_list)
            preds = (scores > 0).astype(int)

        elif model_type == "pnn":
            all_mu, all_sigma, all_alpha = [], [], []
            with torch.no_grad():
                for start in range(0, len(sequences), BATCH_SIZE):
                    end = min(start + BATCH_SIZE, len(sequences))
                    x_b = torch.tensor(
                        np.ascontiguousarray(sequences[start:end]), dtype=torch.float32
                    )
                    x_b = x_b.reshape(end - start, -1).to(DEVICE)
                    mu, sigma, alpha = model(x_b)
                    all_mu.append(mu.cpu().numpy().flatten())
                    all_sigma.append(sigma.cpu().numpy().flatten())
                    all_alpha.append(alpha.cpu().numpy().flatten())

            mu_arr = np.concatenate(all_mu)
            sigma_arr = np.concatenate(all_sigma)
            alpha_arr = np.concatenate(all_alpha)

            spread_seq = spread_raw[SEQ_LENGTH : SEQ_LENGTH + len(mu_arr)]
            if len(spread_seq) < len(mu_arr):
                spread_seq = np.pad(
                    spread_seq, (0, len(mu_arr) - len(spread_seq)), mode="edge"
                )
            spread_seq = np.where(np.abs(spread_seq) > 0, np.abs(spread_seq), 1e-4)

            scores = compute_spoofing_gains_batch(
                mu_arr, sigma_arr, alpha_arr, spread_seq,
                delta_a=SPOOF_DELTA_A, delta_b=SPOOF_DELTA_B,
                Q=SPOOF_Q, q=SPOOF_q,
                fees=SPOOF_FEES, side="ask",
            )
            preds = (scores > 0).astype(int)

        else:  # prae
            x_tensor = torch.tensor(sequences, dtype=torch.float32)
            loader = DataLoader(
                TensorDataset(x_tensor, x_tensor), batch_size=BATCH_SIZE, shuffle=False
            )
            scores_list = []
            with torch.no_grad():
                for batch in loader:
                    x = batch[0].to(DEVICE)
                    rec, _ = model(x, training=False)
                    scores_list.append(
                        torch.sum((x - rec) ** 2, dim=tuple(range(1, x.dim())))
                        .cpu()
                        .numpy()
                    )
            scores = np.concatenate(scores_list)

            rfdr = RollingFalseDiscoveryRate(window_size=RFDR_WINDOW, alpha=RFDR_ALPHA)
            preds = np.zeros(len(scores), dtype=int)
            for i, s in enumerate(scores):
                is_anom, _ = rfdr.process_new_score(float(s))
                preds[i] = int(is_anom)

        results[f"{model_type}_scores"] = scores
        results[f"{model_type}_preds"] = preds

    return results


# ---------------------------------------------------------------------------
# Main loop: iterate over files once, load/compute per year
# ---------------------------------------------------------------------------
all_scores_by_year = {y: {mt: [] for mt in MODEL_TYPES} for y in TRAIN_YEARS}
all_preds_by_year = {y: {mt: [] for mt in MODEL_TYPES} for y in TRAIN_YEARS}
all_period_labels = []
all_feat_values = []
day_boundaries = [0]

for fi, test_file in enumerate(TEST_FILES):
    basename = os.path.basename(test_file)
    df_day, features_day = _load_and_slice(test_file, "xltime", LOB_COLUMNS, MAX_ROWS)
    n_seq = len(features_day) - SEQ_LENGTH

    if n_seq <= 0:
        logger.warning(
            f"File {basename} too small ({len(features_day)} rows, "
            f"need > {SEQ_LENGTH}). Skipping."
        )
        continue

    # Shared data (independent of train year)
    time_frac_day = get_time_frac(df_day)[: len(features_day)]
    period_labels = assign_period(time_frac_day, PERIODS)
    spread_raw = (df_day["ask-price-1"] - df_day["bid-price-1"]).values

    all_period_labels.append(period_labels[SEQ_LENGTH : SEQ_LENGTH + n_seq])
    all_feat_values.append(
        features_day.iloc[SEQ_LENGTH : SEQ_LENGTH + n_seq].reset_index(drop=True)
    )
    day_boundaries.append(day_boundaries[-1] + n_seq)

    for year in TRAIN_YEARS:
        if _is_file_cached(year, basename):
            cached = np.load(_cache_path(year, basename))
            for mt in MODEL_TYPES:
                all_scores_by_year[year][mt].append(cached[f"{mt}_scores"])
                all_preds_by_year[year][mt].append(cached[f"{mt}_preds"])
            logger.info(f"[{year}] Loaded cached: {basename}")
        else:
            logger.info(f"[{year}] Computing: {basename}")
            results = _compute_file_scores(year, features_day, spread_raw)

            for mt in MODEL_TYPES:
                all_scores_by_year[year][mt].append(results[f"{mt}_scores"])
                all_preds_by_year[year][mt].append(results[f"{mt}_preds"])

            # Save to per-file cache
            os.makedirs(_cache_dir(year), exist_ok=True)
            np.savez(_cache_path(year, basename), **results)
            logger.info(f"[{year}] Cached: {basename}")

# Concatenate all files
for year in TRAIN_YEARS:
    for mt in MODEL_TYPES:
        all_scores_by_year[year][mt] = np.concatenate(all_scores_by_year[year][mt])
        all_preds_by_year[year][mt] = np.concatenate(all_preds_by_year[year][mt])

period_labels_seq = np.concatenate(all_period_labels)
feat_values_seq = pd.concat(all_feat_values, ignore_index=True)
n_total = len(period_labels_seq)
logger.info(f"Total: {n_total} sequences from {len(TEST_FILES)} files.")

2026-03-25 14:35:35,219 | WARNING | File 2010-01-01-TOTF.PA-book.parquet too small (0 rows, need > 25). Skipping.
2026-03-25 14:35:35,384 | INFO | [2015] Computing: 2010-01-04-TOTF.PA-book.parquet
2026-03-25 14:38:25,485 | INFO | [2015] Cached: 2010-01-04-TOTF.PA-book.parquet
2026-03-25 14:38:25,490 | INFO | [2017] Computing: 2010-01-04-TOTF.PA-book.parquet
2026-03-25 14:39:37,719 | INFO | [2017] Cached: 2010-01-04-TOTF.PA-book.parquet
2026-03-25 14:39:38,009 | INFO | [2015] Computing: 2010-01-05-TOTF.PA-book.parquet
2026-03-25 14:43:14,278 | INFO | [2015] Cached: 2010-01-05-TOTF.PA-book.parquet
2026-03-25 14:43:14,279 | INFO | [2017] Computing: 2010-01-05-TOTF.PA-book.parquet
2026-03-25 14:44:46,639 | INFO | [2017] Cached: 2010-01-05-TOTF.PA-book.parquet
2026-03-25 14:44:46,780 | INFO | [2015] Computing: 2010-12-29-TOTF.PA-book.parquet


In [ ]:
# Per-year anomaly selection (flagged by >= 1 model)
anom_data = {}
for year in TRAIN_YEARS:
    pred_matrix = np.column_stack([all_preds_by_year[year][mt][:n_total] for mt in MODEL_TYPES])
    n_models_flagged = pred_matrix.sum(axis=1)
    indices = np.where(n_models_flagged >= 1)[0]

    anom_data[year] = {
        "anom_indices": indices,
        "n_anom": len(indices),
        "score_matrix_raw": np.column_stack([all_scores_by_year[year][mt][indices] for mt in MODEL_TYPES]),
        "feat_anom": feat_values_seq.iloc[indices].reset_index(drop=True),
        "period_anom": period_labels_seq[indices],
        "preds_anom": {mt: all_preds_by_year[year][mt][indices] for mt in MODEL_TYPES},
    }

for year in TRAIN_YEARS:
    n_anom = anom_data[year]["n_anom"]
    print(f"Year {year}: {n_anom} anomalies / {n_total} ({100 * n_anom / n_total:.2f}%)")
    for mt in MODEL_TYPES:
        n_mt = int(all_preds_by_year[year][mt][:n_total].sum())
        print(f"  {mt:25s}: {n_mt:6d} ({100 * n_mt / n_total:.2f}%)")
    print()

In [ ]:
SCORE_COLS = ["ocsvm_norm", "pnn_norm", "prae_norm"]
score_data = {}

for year in TRAIN_YEARS:
    score_matrix_full_norm = np.zeros((n_total, len(MODEL_TYPES)), dtype=np.float32)
    for j, mt in enumerate(MODEL_TYPES):
        sc = MinMaxScaler()
        score_matrix_full_norm[:, j] = sc.fit_transform(
            all_scores_by_year[year][mt][:n_total].reshape(-1, 1)
        ).flatten()

    indices = anom_data[year]["anom_indices"]
    score_matrix_norm = score_matrix_full_norm[indices]
    score_df = pd.DataFrame(score_matrix_norm, columns=SCORE_COLS)

    score_data[year] = {
        "score_matrix_norm": score_matrix_norm,
        "score_df": score_df,
        "X_clust": score_df.values,
    }

for year in TRAIN_YEARS:
    print(f"\n--- Year {year} score distribution ---")
    display(score_data[year]["score_df"].describe().round(4))

In [ ]:
cluster_data = {}

for year in TRAIN_YEARS:
    X_clust = score_data[year]["X_clust"]
    n_anom = anom_data[year]["n_anom"]
    preds_anom = anom_data[year]["preds_anom"]

    min_cluster_size = max(5, min(HDBSCAN_MAX_CLUSTER_SIZE, n_anom // HDBSCAN_MIN_CLUSTER_FRAC))
    hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=HDBSCAN_MIN_SAMPLES)
    cluster_labels = hdbscan_model.fit_predict(X_clust)

    unique_clusters = sorted(set(cluster_labels))
    n_hdb_clusters = len(unique_clusters) - (1 if -1 in unique_clusters else 0)
    n_noise = int((cluster_labels == -1).sum())

    # Silhouette & Davies-Bouldin (excluding noise)
    mask_valid = cluster_labels != -1
    if mask_valid.sum() >= 2 and n_hdb_clusters >= 2:
        sil = silhouette_score(X_clust[mask_valid], cluster_labels[mask_valid])
        db = davies_bouldin_score(X_clust[mask_valid], cluster_labels[mask_valid])
    else:
        sil, db = np.nan, np.nan

    # Build cluster names with PNN detection rate
    pnn_rates = {}
    for c in unique_clusters:
        mask = cluster_labels == c
        pnn_rates[c] = preds_anom["pnn"][mask].mean() if mask.sum() > 0 else 0.0

    cluster_names = {}
    for c in unique_clusters:
        if c == -1:
            cluster_names[c] = f"Noise (n={(cluster_labels == c).sum()})"
        else:
            cluster_names[c] = f"C{c} (PNN rate={pnn_rates[c]:.1%})"

    cluster_data[year] = {
        "cluster_labels": cluster_labels,
        "unique_clusters": unique_clusters,
        "n_hdb_clusters": n_hdb_clusters,
        "n_noise": n_noise,
        "cluster_names": cluster_names,
        "min_cluster_size": min_cluster_size,
        "sil": sil,
        "db": db,
    }

    print(f"\n--- Year {year} ---")
    print(f"HDBSCAN: {n_hdb_clusters} clusters, {n_noise} noise ({100*n_noise/n_anom:.1f}%)")
    print(f"  Silhouette: {sil:.4f}, Davies-Bouldin: {db:.4f}")

    mapping_df = pd.DataFrame(
        {
            "cluster": [cluster_names[c] for c in unique_clusters],
            "size": [(cluster_labels == c).sum() for c in unique_clusters],
            "pnn_rate": [round(pnn_rates[c], 4) for c in unique_clusters],
        }
    )
    display(mapping_df)

In [ ]:
CLUSTER_COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(2, len(TRAIN_YEARS), figsize=(7 * len(TRAIN_YEARS), 10))
if len(TRAIN_YEARS) == 1:
    axes = axes.reshape(-1, 1)  # ensure 2D indexing

for col, year in enumerate(TRAIN_YEARS):
    X_clust = score_data[year]["X_clust"]
    cl = cluster_data[year]
    preds_anom = anom_data[year]["preds_anom"]

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_clust)

    # --- Top row: HDBSCAN clusters ---
    ax = axes[0, col]
    for c in cl["unique_clusters"]:
        mask = cl["cluster_labels"] == c
        if c == -1:
            ax.scatter(
                X_pca[mask, 0], X_pca[mask, 1],
                s=6, alpha=0.25, color="lightgrey", label="Noise",
            )
        else:
            ax.scatter(
                X_pca[mask, 0], X_pca[mask, 1],
                s=10, alpha=0.5,
                color=CLUSTER_COLORS[c % len(CLUSTER_COLORS)],
                label=cl["cluster_names"][c],
            )
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    ax.set_title(f"{year} models: HDBSCAN ({cl['n_hdb_clusters']} clusters, min_size={cl['min_cluster_size']})")
    ax.legend(fontsize=6, markerscale=2)

    # --- Bottom row: PNN + Spoofing Gain overlay ---
    ax = axes[1, col]
    pnn_mask = preds_anom["pnn"].astype(bool)
    non_pnn = ~pnn_mask
    ax.scatter(
        X_pca[non_pnn, 0], X_pca[non_pnn, 1],
        s=6, alpha=0.2, color="lightgrey", label="Other anomalies",
    )
    ax.scatter(
        X_pca[pnn_mask, 0], X_pca[pnn_mask, 1],
        s=14, alpha=0.7, color="#c0392b", edgecolors="black", linewidths=0.3,
        label=f"PNN spoofing (n={pnn_mask.sum()})",
    )
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    ax.set_title(f"{year} models: PNN + Spoofing Gain")
    ax.legend(fontsize=7, markerscale=2)

plt.tight_layout()
plt.show()

In [ ]:
max_clusters = max(len(cluster_data[y]["unique_clusters"]) for y in TRAIN_YEARS)
fig, axes_arr = plt.subplots(1, len(TRAIN_YEARS), figsize=(6 * len(TRAIN_YEARS), 0.6 * max_clusters + 2))
if len(TRAIN_YEARS) == 1:
    axes_arr = [axes_arr]

for col, year in enumerate(TRAIN_YEARS):
    cl = cluster_data[year]
    score_matrix_norm = score_data[year]["score_matrix_norm"]
    preds_anom = anom_data[year]["preds_anom"]
    n_anom = anom_data[year]["n_anom"]

    profile_rows = []
    for c in cl["unique_clusters"]:
        mask = cl["cluster_labels"] == c
        row = {
            "Cluster": cl["cluster_names"][c],
            "N": int(mask.sum()),
            "Share (%)": round(100 * mask.sum() / n_anom, 2),
        }
        for j, mt in enumerate(MODEL_TYPES):
            short = mt.replace("transformer_ocsvm", "ocsvm")
            row[f"{short}_score_mean"] = round(float(score_matrix_norm[mask, j].mean()), 4)
            row[f"{short}_det_rate"] = round(float(preds_anom[mt][mask].mean()), 4)
        profile_rows.append(row)

    profile_df = pd.DataFrame(profile_rows).set_index("Cluster")
    print(f"\n--- Year {year} Profile ---")
    display(profile_df)

    det_cols = [c for c in profile_df.columns if c.endswith("_det_rate")]
    det_df = profile_df[det_cols].rename(
        columns={c: c.replace("_det_rate", "") for c in det_cols}
    )
    ax = axes_arr[col]
    sns.heatmap(
        det_df.astype(float),
        annot=True,
        fmt=".2f",
        cmap="YlOrRd",
        vmin=0,
        vmax=1,
        ax=ax,
        linewidths=0.5,
    )
    ax.set_title(f"{year}: Detection rate per model")
    ax.set_xlabel("Model")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
for year in TRAIN_YEARS:
    cl = cluster_data[year]
    preds_anom = anom_data[year]["preds_anom"]
    feat_anom = anom_data[year]["feat_anom"]
    period_anom = anom_data[year]["period_anom"]
    n_anom = anom_data[year]["n_anom"]

    global_mean = feat_anom.mean()
    global_std = feat_anom.std().replace(0, 1e-10)

    summary_rows = []
    for c in cl["unique_clusters"]:
        mask = cl["cluster_labels"] == c
        n_c = int(mask.sum())

        if mask.any():
            period_counts = pd.Series(period_anom[mask]).value_counts()
            dominant_period = period_counts.idxmax() if not period_counts.empty else "-"
            c_mean = feat_anom.iloc[mask].mean()
            top_feat = ((c_mean - global_mean) / global_std).abs().idxmax()
        else:
            dominant_period = "-"
            top_feat = "-"

        pnn_rate = float(preds_anom["pnn"][mask].mean()) if mask.any() else 0.0
        ocsvm_rate = (
            float(preds_anom["transformer_ocsvm"][mask].mean()) if mask.any() else 0.0
        )
        prae_rate = float(preds_anom["prae"][mask].mean()) if mask.any() else 0.0

        if pnn_rate >= 0.5:
            hint = "spoofing-type"
        elif ocsvm_rate >= 0.5 and prae_rate >= 0.5:
            hint = "general (OC-SVM + PRAE)"
        elif ocsvm_rate >= 0.5:
            hint = "general (OC-SVM)"
        elif prae_rate >= 0.5:
            hint = "general (PRAE)"
        else:
            hint = "mixed"

        summary_rows.append(
            {
                "Cluster": cl["cluster_names"][c],
                "N": n_c,
                "Share (%)": round(100 * n_c / n_anom, 2),
                "OC-SVM det.": round(ocsvm_rate, 3),
                "PNN det.": round(pnn_rate, 3),
                "PRAE det.": round(prae_rate, 3),
                "Dominant period": dominant_period,
                "Top feature": top_feat,
                "Interpretation": hint,
            }
        )

    print(f"\n{'='*80}")
    print(f"  Summary for models trained on {year}")
    print(f"{'='*80}")
    summary_df = pd.DataFrame(summary_rows).set_index("Cluster")
    display(summary_df)